# any-llm Agent Tests

Agent-capability parity tests for the any-llm provider (`AnyLLMAgent` — the shared runtime loop over `AnyLLMProvider`), using colon-form any-llm model ids (e.g. `openai:gpt-4o-mini`).

Execution contract:
- **Setup** (the next two code cells) is the only global prerequisite — run those first.
- Every top-level `#` section is **independent** of every other section: each defines its own tools, storage adapters and agent. Cells *within* a section run top-to-bottom.
- Deliberately absent (unsupported by design — see `DESIGN.md`): server tools (web_search / code execution), Anthropic Skills, citations, provider-hosted file generation. Provider-specific features ride the `AnyLLMConfig.api_kwargs` / `client_args` escape hatches (last section).

In [ ]:
from dotenv import load_dotenv
import asyncio
import json
import os

import nest_asyncio

load_dotenv('../../../.env')
nest_asyncio.apply()

# any-llm model ids are colon-form "provider:model".
MODEL = os.getenv('ANY_LLM_AGENT_TEST_MODEL_PRIMARY', 'openai:gpt-4o-mini')
REASONING_MODEL = os.getenv('ANY_LLM_AGENT_TEST_MODEL_REASONING', '')
ENABLE_REASONING = os.getenv('ANY_LLM_ENABLE_REASONING_TEST') == '1'
print(f'Primary model: {MODEL}')

In [ ]:
# ── Test Utils ─────────────────────────────────────────────────────
#
# The loop emits typed StreamItems (content deltas + MetaEnvelope control
# frames) into the agent's Rung-1 stream queue. Like the demo server, we
# re-attach a fresh queue per run and drive the turn with agent.run().

_DONE = object()  # local sentinel: pushed when the run task finishes


def _render_item(item) -> str:
    """Compact, human-readable line for one StreamItem."""
    kind = getattr(item, "kind", None)
    if kind is not None:  # MetaEnvelope control frame
        return f"[meta:{kind}] {item.body}"
    t = getattr(item, "type", "")
    if t == "text":
        return item.text
    if t == "thinking":
        return f"[thinking] {item.thinking}"
    if t == "tool_call":
        return f"[tool_call {item.tool_name}] {item.arguments_json}"
    if t == "tool_result":
        return f"[tool_result {item.tool_name}] {str(item.result_content)[:300]}"
    return repr(item)


def _attach_fresh_queue(agent) -> asyncio.Queue:
    queue: asyncio.Queue = asyncio.Queue()
    agent._stream_queue = queue  # per-run re-attach (same seam the demo server uses)
    return queue


def _start_runner(agent, prompt, queue):
    async def _runner():
        try:
            return await agent.run(prompt)
        finally:
            queue.put_nowait(_DONE)

    return asyncio.create_task(_runner())


async def run_with_stream(agent, prompt, *, print_items: bool = True):
    """Drive one full turn while printing the typed stream; returns AgentResult."""
    queue = _attach_fresh_queue(agent)
    runner = _start_runner(agent, prompt, queue)
    while True:
        item = await queue.get()
        if item is _DONE:
            break
        if print_items:
            print(_render_item(item), flush=True)
        queue.task_done()
    return await runner


async def run_until_pause_or_done(agent, prompt, *, print_items: bool = True):
    """Start a run and drain until it completes OR parks on a frontend-tool
    pause (an `await_input` MetaEnvelope; relay-await §2.3 — the actor stays
    PARKED in RAM and agent.run() does NOT return yet).

    Returns (runner_task, await_input_envelope_or_None).
    """
    queue = _attach_fresh_queue(agent)
    runner = _start_runner(agent, prompt, queue)
    while True:
        item = await queue.get()
        if item is _DONE:
            return runner, None
        if print_items:
            print(_render_item(item), flush=True)
        queue.task_done()
        if getattr(item, "kind", None) == "await_input":
            return runner, item


async def finish_run(agent, runner, *, print_items: bool = True):
    """After submit(ToolReply(...)), drain the still-attached stream until the
    parked turn completes; returns the AgentResult."""
    queue = agent._stream_queue
    while True:
        item = await queue.get()
        if item is _DONE:
            break
        if print_items:
            print(_render_item(item), flush=True)
        queue.task_done()
    return await runner


class DemoHandle:
    """Handle for an in-flight streaming run (abort/steer testing)."""

    def __init__(self, agent, runner, queue, collector):
        self.agent = agent
        self._runner = runner
        self._queue = queue
        self._collector = collector

    async def _cleanup(self):
        if not self._collector.done():
            await self._collector
        # Print anything a steered continuation emitted after the original
        # runner finished (the collector exits at the runner's _DONE).
        while not self._queue.empty():
            item = self._queue.get_nowait()
            if item is not _DONE:
                print(_render_item(item), flush=True)

    async def abort(self):
        result = await self.agent.abort()
        if not self._runner.done():
            await self._runner
        await self._cleanup()
        _print_result(result)
        return result

    async def steer(self, instruction: str):
        result = await self.agent.steer(instruction)
        if not self._runner.done():
            await self._runner
        await self._cleanup()
        _print_result(result)
        return result


def _print_result(result) -> None:
    print(f"stop_reason={result.stop_reason}")
    print(f"was_aborted={getattr(result, 'was_aborted', None)}")
    print(f"abort_phase={getattr(result, 'abort_phase', None)}")
    print(f"total_steps={result.total_steps}")
    print(f"final_answer={result.final_answer!r}")


async def start_demo(agent, prompt: str) -> DemoHandle:
    """Start a streaming run and return a handle for abort/steer."""
    queue = _attach_fresh_queue(agent)
    runner = _start_runner(agent, prompt, queue)

    async def _collect():
        while True:
            item = await queue.get()
            if item is _DONE:
                break
            print(_render_item(item), flush=True)
            queue.task_done()

    collector = asyncio.create_task(_collect())
    return DemoHandle(agent, runner, queue, collector)

# Tool Schema Tests

In [ ]:
from typing import Any

from agent_base.tools import ToolRegistry, tool


@tool
def get_weather(location: str, unit: str = "celsius") -> str:
    """Get the current weather in a given location.

    This function retrieves weather information for the specified location
    and returns it in the requested temperature unit.

    Args:
        location: The city and state, e.g. "San Francisco, CA"
        unit: The unit of temperature, either "celsius" or "fahrenheit"

    Returns:
        String describing the current weather
    """
    # Mock implementation
    return f"The weather in {location} is 22°{unit[0].upper()}"

@tool
def plan_travel_itinerary(
    travelers: list[dict[str, str]],
    destination: str,
    start_date: str,
    end_date: str,
    preferences: dict[str, Any] | None = None,
) -> str:
    """Plan a multi-day travel itinerary.

    This tool accepts structured traveler information along with
    trip preferences and returns a concise summary. It showcases
    nested objects and optional parameters in the schema.

    Args:
        travelers: List of traveler profiles including name and role
        destination: Target city or country for the itinerary
        start_date: Trip start in ISO format (YYYY-MM-DD)
        end_date: Trip end in ISO format (YYYY-MM-DD)
        preferences: Optional mapping of preference category to value

    Returns:
        String summary describing the generated itinerary
    """
    traveler_list = ", ".join(
        f"{traveler['name']} ({traveler.get('role', 'guest')})"
        for traveler in travelers
    )
    pref_summary = (
        ", ".join(f"{key}={value}" for key, value in preferences.items())
        if preferences
        else "standard preferences"
    )
    return (
        f"Trip to {destination} from {start_date} to {end_date} for "
        f"{len(travelers)} traveler(s): {traveler_list}. Preferences: {pref_summary}."
    )


registry = ToolRegistry()
registry.register_tools([get_weather, plan_travel_itinerary])
print("Registered tools:", [schema.name for schema in registry.get_schemas()])

## Export as OpenAI Wire Format

In [ ]:
from agent_base.providers.any_llm import AnyLLMMessageFormatter

wire_schemas = AnyLLMMessageFormatter().format_tool_schemas(registry.get_schemas())
print(json.dumps(wire_schemas, indent=2))

assert all(ws['type'] == 'function' for ws in wire_schemas)
assert {ws['function']['name'] for ws in wire_schemas} == {'get_weather', 'plan_travel_itinerary'}
assert wire_schemas[0]['function']['parameters']['type'] == 'object'

# Basic Agent with Tools

In [ ]:
from agent_base.providers.any_llm import AnyLLMAgent
from agent_base.storage import create_adapters
from agent_base.tools import tool

basic_config, basic_conv, basic_run = create_adapters(
    "filesystem",
    base_path="./test_data/any_llm_basic",
)


@tool
def add(a: int, b: int) -> str:
    """Add two numbers together.

    Args:
        a: First number
        b: Second number

    Returns:
        The sum as a string
    """
    return str(a + b)

@tool
def multiply(a: int, b: int) -> str:
    """Multiply two numbers together.

    Args:
        a: First number
        b: Second number

    Returns:
        The product as a string
    """
    return str(a * b)

SAMPLE_TOOLS = [add, multiply]

agent = AnyLLMAgent(
    system_prompt="You are a helpful assistant that should help the user with their questions.",
    model=MODEL,
    tools=SAMPLE_TOOLS,
    config_adapter=basic_config,
    conversation_adapter=basic_conv,
    run_adapter=basic_run,
)

In [ ]:
await agent.initialize()
agent_uuid = agent.agent_uuid
print(agent_uuid)

## Running w/o queue

In [ ]:
res_wo_queue = await agent.run(prompt="What is 5 + (10 * 2)? Use the tools.")
print(res_wo_queue.final_answer)

assert res_wo_queue.stop_reason == 'end_turn'
assert '25' in res_wo_queue.final_answer
assert res_wo_queue.total_steps >= 2  # at least one tool round-trip

## Running with the Typed Stream

The loop emits typed `StreamItem`s — content deltas (`TextDelta`, `ThinkingDelta`, `ToolCallDelta`, `ToolResultDelta`, ...) plus `MetaEnvelope` control frames (`run_started`, `usage_report`, `await_input`, `run_completed`, ...) — into the agent's single-subscriber stream queue. Wire framing (SSE) is a separate layer (`SseCodec`); here we just render items as text.

In [ ]:
result = await run_with_stream(agent, "Now multiply that result by 3. Use the tools.")
print(result.final_answer)
assert '75' in result.final_answer

### Agent Resume

In [ ]:
agent2 = AnyLLMAgent(
    agent_uuid=agent_uuid,
    tools=SAMPLE_TOOLS,
    config_adapter=basic_config,
    conversation_adapter=basic_conv,
    run_adapter=basic_run,
)
await agent2.initialize()
print(agent2.agent_uuid)

result2 = await run_with_stream(agent2, "What did you do before? Explain briefly.")
print(result2.final_answer)
assert result2.final_answer

# Agent with Answer Check

In [ ]:
import re

from agent_base.core.end_turn_hook import EndTurnContext, EndTurnHookResult
from agent_base.providers.any_llm import AnyLLMAgent


def validate_answer(answer: str) -> tuple[bool, str]:
    """Validate that the answer contains a JSON object with 'name' and 'description'."""
    text = (answer or '').strip()
    if not text:
        return False, 'Input text is empty'

    candidates = re.findall(r'```(?:json)?\s*(\{.*?\})\s*```', text, re.DOTALL)
    candidates += re.findall(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', text, re.DOTALL)
    candidates.append(text)

    for candidate in candidates:
        try:
            parsed = json.loads(candidate.strip())
        except json.JSONDecodeError:
            continue
        if isinstance(parsed, dict) and parsed.get('name') and parsed.get('description'):
            return True, json.dumps(parsed, ensure_ascii=False)
    return False, "No valid JSON with 'name' and 'description' fields found in text"


# The old final_answer_check ctor kwarg is gone -- answer validation now
# rides the end_turn_hook seam: action="retry" with a rollback_message
# bounces the turn back to the model, action="pass" accepts it.
def answer_check_hook(ctx: EndTurnContext) -> EndTurnHookResult:
    ok, detail = validate_answer(ctx.final_text)
    if ok:
        return EndTurnHookResult(action="pass")
    return EndTurnHookResult(
        action="retry",
        rollback_message=(
            f"Your final answer failed validation: {detail} "
            "Reply with ONLY a JSON object containing the fields "
            "'name' and 'description'."
        ),
    )


check_agent = AnyLLMAgent(
    system_prompt=(
        "You are a helpful assistant. Always return the final answer as a JSON "
        "object with exactly the fields 'name' and 'description'."
    ),
    model=MODEL,
    end_turn_hook=answer_check_hook,
)

In [ ]:
result = await run_with_stream(
    check_agent,
    "Invent a fictional gadget for time management. Answer as JSON with the fields 'name' and 'description'.",
)
ok, detail = validate_answer(result.final_answer)
print(result.final_answer)
print(ok, detail)
assert ok

# Multimodal: Image + Client Tool

In [ ]:
import base64
import os

from agent_base.tools import tool

# Handle running from the package dir or the repo root; fall back to a 1x1 PNG
# so the section stays runnable without the fixture image.
test_image = "./currency_receipt_usd_jpy.png"
if not os.path.exists(test_image):
    test_image = "../../../currency_receipt_usd_jpy.png"

if os.path.exists(test_image):
    with open(test_image, "rb") as image_file:
        base64_image = base64.b64encode(image_file.read()).decode("utf-8")
    image_question = "I paid the receipt in USD. What is the amount in INR? Use get_rate."
else:
    base64_image = "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mP8/5+hHgAHggJ/PchI7wAAAABJRU5ErkJggg=="
    image_question = "Describe this image, then convert 10 USD to INR using get_rate."


@tool
def get_rate(source: str, target: str) -> str:
    """Get the exchange rate from source to target currency.

    Args:
        source: The source currency code. It can be INR, USD, or JPY.
        target: The target currency code. It can be INR, USD, or JPY.

    Returns:
        A string describing the exchange rate from source to target currency
    """
    # For testing: return fixed mock rates
    mock_rates = {
        ("INR", "USD"): 0.012,
        ("USD", "INR"): 83.1,
        ("INR", "JPY"): 1.5,
        ("JPY", "INR"): 66.7,
        ("USD", "JPY"): 110.0,
        ("JPY", "USD"): 0.0091,
    }
    return f"The exchange rate from {source} to {target} is {mock_rates[(source, target)]}"

In [ ]:
from agent_base.core.messages import Message
from agent_base.core.types import ImageContent, TextContent
from agent_base.providers.any_llm import AnyLLMAgent

mm_agent = AnyLLMAgent(
    system_prompt="You are a helpful assistant that should help the user with their questions.",
    model=MODEL,
    tools=[get_rate],
)

mm_prompt = Message.user([
    ImageContent(source_type="base64", media_type="image/png", data=base64_image),
    TextContent(text=image_question),
])

mm_result = await run_with_stream(mm_agent, mm_prompt)
print(mm_result.final_answer)
assert mm_result.final_answer

# Agent with Frontend Tools

In [ ]:
from agent_base.tools import tool

# Define a frontend tool - this will be executed by the browser, not the server
@tool(executor="frontend")
def user_confirm(message: str) -> str:
    """Ask the user for yes/no confirmation before proceeding with an action.

    Use this tool when you need explicit user approval before taking an action
    that could have significant consequences.

    Args:
        message: The confirmation message to display to the user, explaining
                what action requires their approval.

    Returns:
        "yes" if user confirms, "no" if user declines
    """
    pass  # Never executed server-side - runs in browser


# Define a backend tool - executed on server
@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression.

    Args:
        expression: A mathematical expression to evaluate (e.g., "2 + 2")

    Returns:
        The result of the calculation as a string
    """
    try:
        # Safe eval for simple math
        allowed = set("0123456789+-*/().% ")
        if all(c in allowed for c in expression):
            result = eval(expression)
            return f"Result: {result}"
        return "Error: Invalid expression"
    except Exception as e:
        return f"Error: {str(e)}"

# Verify tool executor attributes
print(f"user_confirm executor: {user_confirm.__tool_executor__}")
print(f"calculate executor: {calculate.__tool_executor__}")

In [ ]:
from agent_base.providers.any_llm import AnyLLMAgent
from agent_base.storage import create_adapters

# Filesystem storage adapters for persistence across re-hydration
fs_config, fs_conv, fs_run = create_adapters(
    "filesystem",
    base_path="./test_data/any_llm_frontend",
)

# Create agent with both backend and frontend tools
frontend_agent = AnyLLMAgent(
    system_prompt="""You are a helpful assistant. When asked to perform calculations,
    use the calculate tool. When you get a result that seems significant (like any
    number over 50), ask for user confirmation using the user_confirm tool before
    reporting the final answer.""",
    model=MODEL,
    tools=[calculate],  # Backend tools
    frontend_tools=[user_confirm],  # Frontend tools - executed in browser
    config_adapter=fs_config,
    conversation_adapter=fs_conv,
    run_adapter=fs_run,
)

print(frontend_agent)

## Running Agent - Pauses for Frontend Tool

When the model calls a frontend tool (like `user_confirm`), the agent:
1. Executes any backend tools first
2. Persists `pending_relay` (including the pause-level `cid`) to storage
3. Emits an `await_input` MetaEnvelope (`correlation_id == cid`, `expects_reply=True`) and PARKS in RAM on the cid-keyed await table

The turn does NOT end — `agent.run()` returns only after the reply arrives (or an abort). `stop_reason="relay"` is gone.

In [ ]:
runner, pause = await run_until_pause_or_done(frontend_agent, "Calculate 25 * 4 for me.")

assert pause is not None, "expected a frontend-tool pause"
print(f"\nPaused on cid: {pause.correlation_id}")
for call in pause.body.tools:
    print(f"  - {call.tool_name}({call.input}) [tool_use_id={call.tool_use_id}]")

In [ ]:
# Inspect the paused state (persisted for cold resume)
pending_relay = frontend_agent.agent_config.pending_relay
print(f"Has pending relay: {pending_relay is not None}")
if pending_relay:
    print(f"Pause cid: {pending_relay.cid}")  # == pause.correlation_id
    print(f"Pending frontend calls: {len(pending_relay.frontend_calls)}")
    for call in pending_relay.frontend_calls:
        print(f"  - name: {call.name}, tool_id: {call.tool_id}, input: {call.input}")
    print(f"Completed backend results: {len(pending_relay.completed_results)}")
print(f"Current step: {frontend_agent.agent_config.current_step}")

## Resolving the Pause — submit(ToolReply(cid, results))

There is ONE relay primitive: the frontend echoes the `await_input` envelope's `correlation_id` (the cid) in `ToolReply(cid, results)` and tags each per-call result by `tool_use_id`. Submitting on the live instance resolves the parked await in place (hot path); the original `agent.run()` task then runs the turn to completion.

In [ ]:
# Addressing info (the cold-resume section below reuses the same shape)
saved_uuid = frontend_agent.agent_uuid
print(f"Saved UUID: {saved_uuid}")

pending_call = pause.body.tools[0]
tool_use_id = pending_call.tool_use_id
print(f"Pause cid: {pause.correlation_id}")
print(f"Tool use ID: {tool_use_id}")
print(f"Tool name: {pending_call.tool_name}")
print(f"Tool input: {pending_call.input}")

In [ ]:
from agent_base.core.commands import ToolReply
from agent_base.core.types import ToolResultContent

# Simulate the user clicking "yes" in the browser: echo the cid, tag the
# result by tool_use_id. submit() is CQRS — it returns an Ack immediately;
# output keeps flowing on the stream read path.
ack = await frontend_agent.submit(
    ToolReply(
        cid=pause.correlation_id,
        results=[
            ToolResultContent(
                tool_id=tool_use_id,
                tool_name=pending_call.tool_name,
                tool_result="yes",  # User confirmed
                is_error=False,
            )
        ],
    )
)
print(f"Ack: {ack.disposition}")

In [ ]:
# The parked actor woke in place; drain the stream until the turn completes.
result2 = await finish_run(frontend_agent, runner)

print(f"\n\nStop reason: {result2.stop_reason}")
print(f"Total steps: {result2.total_steps}")
print("Final Answer:")
print(result2.final_answer)
assert result2.stop_reason == 'end_turn'

## Cold Resume (process restart) — SessionManager.submit

If the process dies while parked, the pause survives in storage (`pending_relay.cid` is the persisted cold-match key, R23). A restarted process resumes through the SAME front door: `SessionManager.submit(root_session_id, ToolReply(cid, results))` rehydrates the session from storage, re-arms the SAME cid WITHOUT re-emitting `await_input`, resolves it, and re-enters the suspended turn out-of-band (reconcile → splice → checkpoint → resume; relay-await §2.4).

In [ ]:
# Park a fresh run that we will deliberately "lose".
cold_agent = AnyLLMAgent(
    system_prompt=frontend_agent.system_prompt,
    model=frontend_agent.model,
    tools=[calculate],
    frontend_tools=[user_confirm],
    config_adapter=fs_config,
    conversation_adapter=fs_conv,
    run_adapter=fs_run,
)

cold_runner, cold_pause = await run_until_pause_or_done(cold_agent, "Calculate 30 * 3 for me.")
assert cold_pause is not None, "expected a frontend-tool pause"

cold_uuid = cold_agent.agent_uuid
cold_cid = cold_pause.correlation_id
cold_tool_use_id = cold_pause.body.tools[0].tool_use_id
print(f"\nParked: uuid={cold_uuid} cid={cold_cid}")
assert cold_agent.agent_config.pending_relay.cid == cold_cid  # persisted cold-match key

In [ ]:
from agent_base.await_table import AwaitTable, set_await_table
from agent_base.session.manager import SessionManager

# "Process restart": wipe the live await table — the parked coroutine above is
# abandoned mid-await, exactly as if the process had died. Only storage survives.
set_await_table(AwaitTable())


def build_agent(root_session_id: str, principal=None) -> AnyLLMAgent:
    """Principal-aware factory: rebuilds the agent over the same storage."""
    return AnyLLMAgent(
        system_prompt=frontend_agent.system_prompt,
        model=frontend_agent.model,
        tools=[calculate],
        frontend_tools=[user_confirm],
        config_adapter=fs_config,
        conversation_adapter=fs_conv,
        run_adapter=fs_run,
        agent_uuid=root_session_id,  # resume path: loads persisted state
    )


manager = SessionManager(build_agent)

ack = await manager.submit(
    cold_uuid,
    ToolReply(
        cid=cold_cid,
        results=[
            ToolResultContent(
                tool_id=cold_tool_use_id,
                tool_name="user_confirm",
                tool_result="yes",
            )
        ],
    ),
)
print(f"Ack: {ack.disposition}")  # RESOLVED — rehydrate-then-resolve on the SAME cid

In [ ]:
# The RESOLVED reply kicked the out-of-band continuation; await it for the
# final result. (A server would attach a stream queue before submitting and
# stream until the run_completed frame — see demos/fastapi_server.)
rehydrated = await manager.get_or_create(cold_uuid)
result_cold = await rehydrated._rearmed_resume_task

print(f"Stop reason: {result_cold.stop_reason}")
print("Final Answer:")
print(result_cold.final_answer)

# Subagent Test

## Travel Planning with Specialist Subagents
Orchestrator delegates flight searches to `flight_finder` and hotel searches to `hotel_finder`, then synthesizes results.

In [ ]:
# Define specialist tools for subagents
from agent_base.tools import tool

@tool
def search_flights(origin: str, destination: str, date: str) -> str:
    """Search for available flights between two cities on a given date.

    Args:
        origin: The departure city, e.g. "New York"
        destination: The arrival city, e.g. "London"
        date: The travel date in YYYY-MM-DD format

    Returns:
        A string listing available flights with prices
    """
    flights = {
        ("New York", "Tokyo"): [
            {"airline": "ANA", "departure": "10:00", "arrival": "14:00+1", "price": "$1,200"},
            {"airline": "JAL", "departure": "13:30", "arrival": "17:30+1", "price": "$1,350"},
        ],
        ("Tokyo", "New York"): [
            {"airline": "ANA", "departure": "17:00", "arrival": "16:00", "price": "$1,180"},
            {"airline": "United", "departure": "19:30", "arrival": "18:30", "price": "$1,050"},
        ],
    }
    key = (origin, destination)
    if key not in flights:
        return f"No flights found from {origin} to {destination} on {date}."
    results = [f"Flights from {origin} to {destination} on {date}:"]
    for f in flights[key]:
        results.append(f"  - {f['airline']}: departs {f['departure']}, arrives {f['arrival']}, {f['price']}")
    return "\n".join(results)


@tool
def search_hotels(city: str, checkin: str, checkout: str) -> str:
    """Search for available hotels in a city for given dates.

    Args:
        city: The city to search hotels in, e.g. "Tokyo"
        checkin: Check-in date in YYYY-MM-DD format
        checkout: Check-out date in YYYY-MM-DD format

    Returns:
        A string listing available hotels with nightly rates
    """
    hotels = {
        "Tokyo": [
            {"name": "Park Hyatt Tokyo", "rating": "5-star", "price": "$450/night"},
            {"name": "Shinjuku Granbell", "rating": "3-star", "price": "$120/night"},
            {"name": "Hotel Gracery Shinjuku", "rating": "4-star", "price": "$200/night"},
        ],
    }
    if city not in hotels:
        return f"No hotels found in {city} for {checkin} to {checkout}."
    results = [f"Hotels in {city} ({checkin} to {checkout}):"]
    for h in hotels[city]:
        results.append(f"  - {h['name']} ({h['rating']}): {h['price']}")
    return "\n".join(results)

In [ ]:
from agent_base.providers.any_llm import AnyLLMAgent
from agent_base.storage import create_adapters

# Filesystem storage adapters
sa_config, sa_conv, sa_run = create_adapters(
    "filesystem",
    base_path="./test_data/any_llm_subagents",
)
await sa_config.connect()
await sa_conv.connect()
await sa_run.connect()

# Specialist 1: Flight Finder
flight_agent = AnyLLMAgent(
    system_prompt=(
        "You are a flight search specialist. Use the search_flights tool to find "
        "flights for the user. Always search for the exact origin, destination, and "
        "date provided. Present the results clearly."
    ),
    description="Searches for available flights between cities on a specific date",
    model=MODEL,
    tools=[search_flights],
    config_adapter=sa_config,
    conversation_adapter=sa_conv,
    run_adapter=sa_run,
)

# Specialist 2: Hotel Finder
hotel_agent = AnyLLMAgent(
    system_prompt=(
        "You are a hotel search specialist. Use the search_hotels tool to find "
        "hotels for the user. Always search for the exact city and dates provided. "
        "Present the results clearly."
    ),
    description="Searches for available hotels in a city for specific check-in/check-out dates",
    model=MODEL,
    tools=[search_hotels],
    config_adapter=sa_config,
    conversation_adapter=sa_conv,
    run_adapter=sa_run,
)

# Orchestrator: delegates to specialists via SubAgentTool
orchestrator = AnyLLMAgent(
    system_prompt=(
        "You are a travel planning coordinator. When the user asks to plan a trip, "
        "you MUST delegate flight searches to the flight_finder subagent and hotel "
        "searches to the hotel_finder subagent. After receiving results from both, "
        "synthesize them into a concise travel plan with total estimated costs."
    ),
    model=MODEL,
    subagents={
        "flight_finder": flight_agent,
        "hotel_finder": hotel_agent,
    },
    config_adapter=sa_config,
    conversation_adapter=sa_conv,
    run_adapter=sa_run,
)

await orchestrator.initialize()
print(f"Orchestrator UUID: {orchestrator.agent_uuid}")

In [ ]:
prompt = (
    "Plan a 5-night trip from New York to Tokyo. "
    "I need flights departing 2025-03-15 and returning 2025-03-20, "
    "plus a hotel for those dates. Find the best options and give me a total cost estimate."
)

result = await run_with_stream(orchestrator, prompt)

In [ ]:
# Show the final synthesized travel plan
print("=== Final Answer ===")
print(result.final_answer)
assert result.final_answer

In [ ]:
# Inspect execution details
print(f"Stop reason: {result.stop_reason}")
print(f"Total steps: {result.total_steps}")
print(f"Model: {result.model}")
# cost / cumulative_usage are DELETED from AgentResult (O14d) — the per-turn
# billing fact rides on result.settlement (TurnSettlement); cumulative totals
# are the SettlementAggregator's job, fed by usage_report meta frames.
if result.settlement:
    print(f"Turn usage: {result.settlement.turn_usage.totals_dict()}")
    print(f"Turn cost: {result.settlement.turn_cost.to_dict()}")

# Context Externalization Tests

Test file-backed context externalization with strict 10,000-token budgets. These scenarios still keep regular long-chat compaction enabled, but the assertions here focus on `.context/` references for oversized newest-message payloads.

In [ ]:
from functools import lru_cache
from urllib.request import urlopen
from pprint import pprint

from agent_base.core.types import TextContent, ToolResultContent
from agent_base.providers.any_llm import AnyLLMAgent, ExternalizationConfig
from agent_base.providers.any_llm.compaction import CompactionConfig
from agent_base.storage import create_adapters
from agent_base.tools import tool


# Keep the externalization section runnable after a fresh kernel restart.
config_adapter, conversation_adapter, run_adapter = create_adapters("memory")

LARGE_DOCUMENT_TITLE = "War and Peace"
LARGE_DOCUMENT_URL = "https://www.gutenberg.org/cache/epub/2600/pg2600.txt"
FALLBACK_DOCUMENT_TEXT = """
BOOK ONE: 1805

"Well, Prince, so Genoa and Lucca are now just family estates of the Buonapartes.
But I warn you, if you don't tell me that this means war, if you still try to defend
the infamies and horrors perpetrated by that Antichrist-I really believe he is Antichrist-
I will have nothing more to do with you and you are no longer my friend, no longer my
'faithful slave,' as you call yourself!"

Anna Pavlovna's words came with composure and confidence. She spoke in French, as she
always did when feeling especially animated, and her familiar visitor answered in the
same language and tone as though the whole of Europe could be neatly arranged between
one drawing-room call and the next.
""".strip()


@lru_cache(maxsize=1)
def load_large_document() -> str:
    """Fetch and cache a large public-domain document for compaction demos."""
    try:
        with urlopen(LARGE_DOCUMENT_URL, timeout=20) as response:
            raw_text = response.read().decode("utf-8")
    except Exception:
        raw_text = FALLBACK_DOCUMENT_TEXT

    start_marker = f"*** START OF THE PROJECT GUTENBERG EBOOK {LARGE_DOCUMENT_TITLE.upper()} ***"
    end_marker = f"*** END OF THE PROJECT GUTENBERG EBOOK {LARGE_DOCUMENT_TITLE.upper()} ***"
    if start_marker in raw_text:
        raw_text = raw_text.split(start_marker, 1)[1].lstrip()
    if end_marker in raw_text:
        raw_text = raw_text.split(end_marker, 1)[0].rstrip()
    return raw_text


def make_filler(tokens: int = 12_000) -> str:
    """Return an approximately sized excerpt from a cached real document."""
    header = f"Document: {LARGE_DOCUMENT_TITLE}\nSource: {LARGE_DOCUMENT_URL}\n\n"
    target_chars = max(tokens * 4 - len(header), 0)
    document = load_large_document()
    if len(document) < target_chars:
        repeats = (target_chars // max(len(document), 1)) + 1
        document = ((document + "\n\n") * repeats).strip()
    return header + document[:target_chars]


@tool
def large_result_tool(query: str) -> str:
    """Look up information about a topic. Returns detailed results.

    Args:
        query: The search query

    Returns:
        Detailed information about the query
    """
    return f"Result for '{query}': " + make_filler(12_000)


@tool
def medium_result_tool(topic: str) -> str:
    """Fetch a medium-sized report on a topic.

    Args:
        topic: The topic to report on

    Returns:
        A medium-length report
    """
    # ~4k tokens each — individually under the per-result threshold,
    # but 3 parallel calls = ~12k tokens which exceeds the combined budget.
    return f"Report on '{topic}': " + make_filler(4_000)


COMPACTION_TOOLS = [large_result_tool, medium_result_tool]

STRICT_COMPACTION = CompactionConfig(
    threshold_tokens=5000,
    preserve_recent_tokens=2_000,
)

STRICT_EXTERNALIZATION = ExternalizationConfig(
    max_prompt_tokens=10_000,
    max_tool_result_tokens=10_000,
    max_combined_tool_result_tokens=10_000,
)


def first_tool_result_message(messages):
    return next(
        msg for msg in messages
        if any(isinstance(block, ToolResultContent) for block in msg.content)
    )

## Scenario 1: User Prompt too large

In [ ]:
from agent_base.common_tools import GlobFileSearchTool, ListDirTreeTool, ReadFileTool

# Scenario 1: Initial user prompt is too large and should be externalized
agent_compact = AnyLLMAgent(
    system_prompt="You are a helpful assistant.",
    model=MODEL,
    compaction_config=STRICT_COMPACTION,
    externalization_config=STRICT_EXTERNALIZATION,
    tools=[
        ReadFileTool().as_tool(),
        ListDirTreeTool().as_tool(),
        GlobFileSearchTool().as_tool(),
    ] + COMPACTION_TOOLS,
    config_adapter=config_adapter,
    conversation_adapter=conversation_adapter,
    run_adapter=run_adapter,
)
await agent_compact.initialize()

big_prompt = (
    "Please summarize the following text:\n\n"
    + make_filler(12_000)
)

result_1 = await run_with_stream(agent_compact, big_prompt)
print(f"\n--- Scenario 1 Results ---")
print(f"Stop reason: {result_1.stop_reason}")
print(f"Total steps: {result_1.total_steps}")
print(result_1.final_answer)

In [ ]:
print(agent_compact.agent_config.last_known_input_tokens)
print(agent_compact.agent_config.last_known_output_tokens)
pprint(result_1.settlement)

In [ ]:
from IPython.lib.display import FileLink
import glob

sandbox_dir = agent_compact._sandbox.root
prompt_files = sorted(glob.glob(f"{sandbox_dir}/.context/prompt_*.txt"))
assert prompt_files, 'expected the oversized prompt to be externalized to .context/'
FileLink(prompt_files[-1])

## Scenario 2: Large Tool Result

In [ ]:
result_2 = await run_with_stream(
    agent_compact,
    "Look up information on Leo Tolstoy's War and Peace. Use the large_result_tool to look up "
        "information. After getting results, provide a brief summary."
)
print(f"\n--- Scenario 2 Results ---")
print(f"Stop reason: {result_2.stop_reason}")
print(f"Total steps: {result_2.total_steps}")
print(result_2.final_answer)

In [ ]:
context_tool_result_msg_2 = first_tool_result_message(agent_compact.agent_config.context_messages)

context_block_2 = next(
    block for block in context_tool_result_msg_2.content
    if isinstance(block, ToolResultContent)
)
sandbox_dir = agent_compact._sandbox.root
expected_tool_result_path_2 = f"{sandbox_dir}/.context/tool_result_{context_block_2.tool_id}.txt"
FileLink(expected_tool_result_path_2)

## Scenario 3: Parallel Tool Calls overflow context

In [ ]:
result_3 = await run_with_stream(
    agent_compact,
    "Compare these three topics: leo tolstoy's war and peace, the index of the book, and the first chapter of the book."
    "Look up each one using the medium_result_tool. ALWAYS use the medium_result_tool for EACH topic IN PARALLEL."
)
print(f"\n--- Scenario 3 Results ---")
print(f"Stop reason: {result_3.stop_reason}")
print(f"Total steps: {result_3.total_steps}")
print(result_3.final_answer)

In [ ]:
for msg in agent_compact.agent_config.context_messages:
    print(msg.role)
    for content_block in msg.content:
        if content_block.content_block_type == "text":
            content = content_block.text
        elif content_block.content_block_type == "thinking":
            content = content_block.thinking
        else:
            content = content_block.to_dict()
        print(content_block.content_block_type, content)
    print("\n---\n")

# Steer and Abort

## Streaming Abort and Steer

In [ ]:
from agent_base.providers.any_llm import AnyLLMAgent, AnyLLMConfig

steer_agent = AnyLLMAgent(
    system_prompt=(
        "You are a helpful assistant. Think carefully and answer in a detailed, "
        "step-by-step way."
    ),
    model=MODEL,
    config=AnyLLMConfig(max_tokens=4096),
)

steer_prompt = ("Explain in a very detailed way how to solve 5 + (10 * 2). "
    "Think first, then explain every intermediate step slowly.")

In [ ]:
demo = await start_demo(steer_agent, steer_prompt)

In [ ]:
res_abort = await demo.abort()
assert res_abort.stop_reason == 'aborted'

In [ ]:
# The aborted turn left a clean chain; a fresh run continues the session.
res_continue = await steer_agent.run("Explain the BODMAS rule first, then solve the expression.")
print(res_continue.final_answer)

In [ ]:
demo = await start_demo(steer_agent, steer_prompt)

In [ ]:
res_steer = await demo.steer("Give only a very concise explanation.")
print(res_steer.final_answer)

## Tool Call Abort and Steer

In [ ]:
from agent_base.providers.any_llm import AnyLLMAgent, AnyLLMConfig
from agent_base.tools import tool

@tool
async def slow_calculate(expression: str, delay_seconds: int = 10) -> str:
    """Evaluate a mathematical expression after a short delay."""
    await asyncio.sleep(delay_seconds)
    return str(eval(expression, {"__builtins__": {}}, {}))

tool_abort_agent = AnyLLMAgent(
    system_prompt=(
        "You are a helpful assistant. Always use the slow_calculate tool for "
        "math questions before answering."
    ),
    model=MODEL,
    config=AnyLLMConfig(max_tokens=4096),
    tools=[slow_calculate],
)

In [ ]:
demo = await start_demo(tool_abort_agent, "Use slow_calculate to compute 25 * 4 with delay_seconds=10, "
            "then explain the result in a few sentences.")

In [ ]:
res_tool_abort = await demo.abort()
assert res_tool_abort.stop_reason == 'aborted'

In [ ]:
res_tool_continue = await run_with_stream(tool_abort_agent, "Continue your work. Call the slow calculate again.")
_print_result(res_tool_continue)
assert '100' in res_tool_continue.final_answer

## Relay Abort and Steer

In [ ]:
from agent_base.providers.any_llm import AnyLLMAgent, AnyLLMConfig
from agent_base.tools import tool

@tool
def quick_calculate(expression: str) -> str:
    """Evaluate a mathematical expression."""
    return str(eval(expression, {"__builtins__": {}}, {}))


@tool(executor="frontend")
def manual_confirm(message: str) -> str:
    """Ask the user for confirmation before continuing."""
    return f"Confirmation requested: {message}"


relay_abort_agent = AnyLLMAgent(
    system_prompt=(
        "You are a helpful assistant. Use quick_calculate for math. If the result "
        "is greater than 50, ask for confirmation with manual_confirm before "
        "giving the final answer."
    ),
    model=MODEL,
    config=AnyLLMConfig(max_tokens=4096),
    tools=[quick_calculate],
    frontend_tools=[manual_confirm],
)

In [ ]:
demo = await start_demo(relay_abort_agent, "Calculate 25 * 4 for me.")

In [ ]:
res_relay_abort = await demo.abort()
assert res_relay_abort.stop_reason == 'aborted'

In [ ]:
# A bare `await agent.run(...)` is no longer pause-safe here: with
# manual_confirm registered the loop PARKS on await_external when the model
# asks for confirmation, so run() would never return (relay-await semantics).
# Drive the turn pause-aware and approve the confirmation if it pauses.
from agent_base.core.commands import ToolReply
from agent_base.core.types import ToolResultContent

runner, pause = await run_until_pause_or_done(
    relay_abort_agent,
    "First tell me what made you stop. Then regardless you have my permission "
    "to execute any calculations, then tell me the result.",
)
if pause is not None:
    ack = await relay_abort_agent.submit(ToolReply(
        cid=pause.correlation_id,
        results=[
            ToolResultContent(
                tool_id=call.tool_use_id,
                tool_name=call.tool_name,
                tool_result="yes",  # user approves
                is_error=False,
            )
            for call in pause.body.tools
        ],
    ))
    print(f"Ack: {ack.disposition}")
    res_relay_continue = await finish_run(relay_abort_agent, runner)
else:
    res_relay_continue = await runner
print(res_relay_continue.final_answer)

# Optional Reasoning

`reasoning_effort` is the harmonised any-llm reasoning knob (first-class on `AnyLLMConfig`); raw provider-specific thinking dicts would ride `api_kwargs` instead.

In [ ]:
from agent_base.providers.any_llm import AnyLLMAgent, AnyLLMConfig

if not (ENABLE_REASONING and REASONING_MODEL):
    print('Skipping reasoning test: set ANY_LLM_ENABLE_REASONING_TEST=1 and ANY_LLM_AGENT_TEST_MODEL_REASONING.')
else:
    reasoning_agent = AnyLLMAgent(
        system_prompt='You are a careful reasoner.',
        model=REASONING_MODEL,
        config=AnyLLMConfig(max_tokens=2048, reasoning_effort='low'),
    )
    reasoning_result = await run_with_stream(reasoning_agent, 'What is 25 * 37? Reason it out.')
    print(reasoning_result.final_answer)
    assert '925' in reasoning_result.final_answer

# Escape Hatch: Provider-Specific api_kwargs

The agent-level view of the escape-hatch pattern: anything not first-class on `AnyLLMConfig` rides `api_kwargs` (merged verbatim into every request, applied last) or `client_args` (provider client constructor). This replaces the per-feature config fields the Anthropic provider exposes.

In [ ]:
from agent_base.providers.any_llm import AnyLLMAgent, AnyLLMConfig

hatch_agent = AnyLLMAgent(
    system_prompt='You are a test bot. Follow instructions exactly.',
    model=MODEL,
    config=AnyLLMConfig(
        max_tokens=64,
        api_kwargs={'temperature': 0},
        client_args={'timeout': 60},
    ),
)

hatch_result = await run_with_stream(hatch_agent, 'Reply with exactly: PONG')
print(hatch_result.final_answer)
assert 'PONG' in hatch_result.final_answer